In [ ]:
# Project-root setup (auto-injected during notebook reorg 2026-05-06).
# Walks up from the notebook's working directory until it finds pyproject.toml,
# then adds that directory to sys.path so `from twopoint_lockin import ...` etc. work
# regardless of which subfolder the notebook lives in. PROJECT_ROOT is also exposed
# so cells below can build absolute paths to data/ and the vendored qickdawg package.
import sys as _sys
from pathlib import Path as _Path
_p = _Path.cwd().resolve()
while _p != _p.parent and not (_p / "pyproject.toml").exists():
    _p = _p.parent
PROJECT_ROOT = _p if (_p / "pyproject.toml").exists() else _Path.cwd()
if str(PROJECT_ROOT) not in _sys.path:
    _sys.path.insert(0, str(PROJECT_ROOT))
print(f"PROJECT_ROOT: {PROJECT_ROOT}")


# Rabi Oscillations & Vector Magnetometry — With Laser Noise Cancellation

**Enhanced version of `02_rabi_vector_magnetometry.ipynb`.**  
Every acquisition runs on **both** ADC channels and the noise correction
$S_{\text{corr}} = S_{\text{ch1}} - \alpha\,S_{\text{ch0}}$
is applied before fitting or field reconstruction.

**Prerequisites:** Complete `01b_noise_cancelled_nv_testing.ipynb` first and bring:
- `RESONANCE_FREQ_MHZ`, `READOUT_OFFSET_TREG`, `READOUT_INTEGRATION_TUS`, `LASER_ON_TUS`, `MW_READOUT_DELAY_TREG`
- **`ALPHA`** — laser noise gain factor calibrated in Notebook 1b, Section 3

**What this notebook covers:**
- Section 0 — Setup & Master Configuration
- Section 1 — Noise-Corrected Rabi Oscillations
- Section 2 — Vector Magnetometry Theory
- Section 3 — Noise-Corrected Wide ODMR (all 8 peaks)
- Section 4 — Peak Assignment to NV Crystallographic Axes
- Section 5 — Frequency Shift Tracking & Field Reconstruction
- Section 6 — Drift Rejection via Field Flipping (optional)
- Section 7 — Error Estimation
- Section 8 — Live Noise-Corrected Field Monitoring

**Hardware setup:**
| Channel | Role |
|---------|------|
| ADC Channel 1 (C) | NV photoluminescence |
| ADC Channel 0 (A) | Laser intensity noise reference |
| MW Channel 0 | Microwave output |
| `laser_gate_pmod = 0` | Placeholder — not wired to laser |

---
## Section 0: Setup & Master Configuration

In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
import matplotlib.pyplot as plt
from copy import copy
import qickdawg as qd
from scipy.optimize import curve_fit
from scipy.linalg import pinv
from scipy.signal import find_peaks as scipy_find_peaks

In [ ]:
# ===== EDIT THIS: Set your RFSoC IP address =====
RFSOC_IP = '172.16.26.5'

qd.start_client(RFSOC_IP)
print(f"Connected to RFSoC at {RFSOC_IP}")

In [ ]:
# ===== MASTER CONFIGURATION =====
# Update all values from your Notebook 1b results.

# ── ADC channels ──────────────────────────────────────────────────────────
NV_CHANNEL    = 1   # Channel C: NV PL
NOISE_CHANNEL = 0   # Channel A: laser noise reference

# ── Noise cancellation gain (from Notebook 1b, Section 3) ─────────────────
ALPHA = 1.0         # <-- update from 01b calibration

# ── Calibrated readout parameters (from Notebook 1b, Section 6) ───────────
RESONANCE_FREQ_MHZ      = 2870.0  # <-- From noise-corrected ODMR
READOUT_OFFSET_TREG     = 0
READOUT_INTEGRATION_TUS = 2.0
LASER_ON_TUS            = 15.0
MW_READOUT_DELAY_TREG   = 35
READOUT_REF_START_TUS   = 12.0

# ── Hardware channel factory ───────────────────────────────────────────────
def make_config(adc_ch=NV_CHANNEL):
    cfg = qd.NVConfiguration()
    cfg.adc_channel     = adc_ch
    cfg.mw_channel      = 0
    cfg.mw_nqz          = 1
    cfg.mw_gain         = 30000
    cfg.laser_gate_pmod = 0
    cfg.relax_delay_tus = 0.5
    return cfg

# ── Noise correction helpers ───────────────────────────────────────────────
def correct_signal_ref(sig_nv, ref_nv, sig_n0, ref_n0, alpha=None):
    """Apply noise correction to signal and reference arrays.

    Returns corrected (signal, reference, contrast_percent).
    Works element-wise for both scalar and array inputs.
    """
    if alpha is None:
        alpha = ALPHA
    sig_c = sig_nv - alpha * sig_n0
    ref_c = ref_nv - alpha * ref_n0
    with np.errstate(divide='ignore', invalid='ignore'):
        contrast_c = np.where(
            np.abs(ref_c) > 0,
            (ref_c - sig_c) / ref_c * 100,
            0.0
        )
    return sig_c, ref_c, contrast_c


def acquire_corrected_odmr(prog_nv, prog_n0):
    """Acquire LockinODMR on both channels and return noise-corrected contrast.

    Returns dict with keys: frequencies, signal, reference, contrast_percent,
    signal_raw, reference_raw, contrast_raw.
    """
    d_nv = prog_nv.acquire(progress=False)
    d_n0 = prog_n0.acquire(progress=False)
    sig_c, ref_c, contrast_c = correct_signal_ref(
        d_nv.signal, d_nv.reference,
        d_n0.signal, d_n0.reference
    )
    return {
        'frequencies':      d_nv.frequencies,
        'signal':           sig_c,
        'reference':        ref_c,
        'contrast_percent': contrast_c,
        'signal_raw':       d_nv.signal,
        'reference_raw':    d_nv.reference,
        'contrast_raw':     d_nv.contrast_percent,
        'noise_signal':     d_n0.signal,
        'noise_reference':  d_n0.reference,
    }


print("Master configuration loaded.")
print(f"  NV channel:             {NV_CHANNEL}  (Channel C)")
print(f"  Noise channel:          {NOISE_CHANNEL}  (Channel A)")
print(f"  Noise gain α:           {ALPHA}")
print(f"  Resonance frequency:    {RESONANCE_FREQ_MHZ} MHz")
print(f"  Readout integration:    {READOUT_INTEGRATION_TUS} μs")
print(f"  Laser on time:          {LASER_ON_TUS} μs")

---
## Section 1: Noise-Corrected Rabi Oscillations

RabiSweep is run on **both** channels with identical timing.  
Noise-corrected contrast:
$$\text{contrast}_{\text{corr}}(t) = \frac{(\text{ref}_{\text{ch1}} - \alpha\,\text{ref}_{\text{ch0}}) - (\text{sig}_{\text{ch1}} - \alpha\,\text{sig}_{\text{ch0}})}{\text{ref}_{\text{ch1}} - \alpha\,\text{ref}_{\text{ch0}}} \times 100\,\%$$

This suppresses laser amplitude noise from the Rabi envelope, giving a cleaner decay envelope and more accurate π/2 pulse extraction.

In [ ]:
# Show the RabiSweep pulse sequence diagram
qd.RabiSweep.plot_sequence()

In [ ]:
# ── Rabi configuration (shared settings) ──────────────────────────────────
def make_rabi_config(adc_ch):
    cfg = make_config(adc_ch)
    cfg.mw_fMHz                      = RESONANCE_FREQ_MHZ
    cfg.mw_gain                      = 30000
    cfg.pre_init                     = True
    cfg.laser_on_tus                 = LASER_ON_TUS
    cfg.readout_integration_tus      = READOUT_INTEGRATION_TUS
    cfg.readout_reference_start_tus  = READOUT_REF_START_TUS
    cfg.mw_readout_delay_treg        = MW_READOUT_DELAY_TREG
    cfg.laser_readout_offset_treg    = READOUT_OFFSET_TREG
    cfg.reps                         = 1000
    # Sweep MW pulse length — adjust stop if Rabi period is longer
    cfg.add_linear_sweep('mw', 'treg', start=4, stop=800, delta=4)
    return cfg

prog_rabi_nv = qd.RabiSweep(make_rabi_config(NV_CHANNEL))
prog_rabi_n0 = qd.RabiSweep(make_rabi_config(NOISE_CHANNEL))

cfg_tmp = make_rabi_config(NV_CHANNEL)
print(f"Rabi sweep: {cfg_tmp.mw_start_tns:.0f} → {cfg_tmp.mw_end_tns:.0f} ns")
print(f"  {cfg_tmp.nsweep_points} points, 1000 reps")
print(f"  Estimated time (NV ch): {prog_rabi_nv.total_time():.1f} s")
print()
print("[1/2] Acquiring Rabi on NV channel...")
d_rabi_nv = prog_rabi_nv.acquire(progress=True)
print("[2/2] Acquiring Rabi on noise channel...")
d_rabi_n0 = prog_rabi_n0.acquire(progress=True)
print("Done.")

In [ ]:
# ── Apply noise correction ─────────────────────────────────────────────────
rabi_sig_c, rabi_ref_c, rabi_contrast_c = correct_signal_ref(
    d_rabi_nv.signal, d_rabi_nv.reference,
    d_rabi_n0.signal, d_rabi_n0.reference
)

t_ns = d_rabi_nv.sweep_tus * 1000   # pulse duration in ns
t_us = d_rabi_nv.sweep_tus

print("Noise correction applied to Rabi data.")
print(f"  Contrast noise std — raw:       {np.std(d_rabi_nv.contrast_percent):.4f} %")
print(f"  Contrast noise std — corrected: {np.std(rabi_contrast_c):.4f} %")

In [ ]:
# ── Plot raw vs noise-corrected Rabi ──────────────────────────────────────
fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)

ax = axes[0]
ax.plot(t_ns, d_rabi_nv.contrast_percent, color='steelblue', linewidth=1.2, alpha=0.7, label='Raw (Ch1 only)')
ax.plot(t_ns, rabi_contrast_c,            color='green',     linewidth=1.5,            label='Noise-corrected')
ax.set_ylabel('Contrast (%)')
ax.set_title('Rabi Oscillations — raw vs noise-corrected')
ax.legend()
ax.grid(True, alpha=0.3)

ax = axes[1]
ax.plot(t_ns, d_rabi_n0.contrast_percent, color='darkorange', linewidth=1.2, alpha=0.8,
        label=f'Noise reference (Ch{NOISE_CHANNEL}) — should be ~0')
ax.axhline(0, color='k', linewidth=0.5, linestyle='--')
ax.set_ylabel('Contrast (%)')
ax.set_xlabel('MW Pulse Duration (ns)')
ax.set_title('Noise Reference Channel — Rabi')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# ── Fit noise-corrected Rabi oscillations ─────────────────────────────────
def decaying_cosine(t, A, T_rabi, tau_decay, offset):
    """Decaying cosine: A * cos(2π t / T_rabi) * exp(-t / tau_decay) + offset.
    t in μs.
    """
    return A * np.cos(2 * np.pi * t / T_rabi) * np.exp(-t / tau_decay) + offset

x = rabi_contrast_c   # use noise-corrected contrast
A0   = (np.max(x) - np.min(x)) / 2
T0   = 0.4    # μs  — adjust if fit fails
tau0 = 1.0    # μs
off0 = np.mean(x)

try:
    popt, _ = curve_fit(decaying_cosine, t_us, x, p0=[A0, T0, tau0, off0], maxfev=10000)
    A_fit, T_rabi_us, tau_us, offset_fit = popt

    pi2_pulse_ns = T_rabi_us / 4 * 1000
    pi_pulse_ns  = T_rabi_us / 2 * 1000

    fig, ax = plt.subplots(figsize=(12, 4))
    ax.plot(t_ns, x, color='green', linewidth=1.5, label='Noise-corrected data')
    ax.plot(t_ns, decaying_cosine(t_us, *popt), 'r--', linewidth=2,
            label=f'Fit  T_Rabi={T_rabi_us*1000:.0f} ns  τ={tau_us*1000:.0f} ns')
    ax.axvline(pi2_pulse_ns, color='darkgreen',  linestyle=':', linewidth=1.5,
               label=f'π/2 = {pi2_pulse_ns:.0f} ns')
    ax.axvline(pi_pulse_ns,  color='darkorange', linestyle=':', linewidth=1.5,
               label=f'π = {pi_pulse_ns:.0f} ns')
    ax.set_xlabel('MW Pulse Duration (ns)')
    ax.set_ylabel('Contrast (%)')
    ax.set_title('Noise-Corrected Rabi Oscillations — Fit')
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

    print("Rabi fit results (noise-corrected):")
    print(f"  Rabi period:   {T_rabi_us*1000:.0f} ns")
    print(f"  π/2 pulse:     {pi2_pulse_ns:.0f} ns  ← update cell below")
    print(f"  π pulse:       {pi_pulse_ns:.0f} ns")
    print(f"  Decay time:    {tau_us*1000:.0f} ns")

except RuntimeError:
    print("Curve fit failed.")
    print("Try adjusting the initial guesses (A0, T0, tau0) based on the raw plot.")

In [ ]:
# ===== EDIT THIS: Save calibrated π/2 pulse length =====
PI2_PULSE_NS = 100   # <-- Update from noise-corrected Rabi fit above

print(f"π/2 pulse: {PI2_PULSE_NS} ns")
print(f"π pulse:   {PI2_PULSE_NS * 2} ns")

In [ ]:
# Live noise-corrected Rabi monitor
# Press Stop (■) to exit.

def get_corrected_rabi():
    d_nv_ = prog_rabi_nv.acquire(progress=False)
    d_n0_ = prog_rabi_n0.acquire(progress=False)
    _, _, contrast_c = correct_signal_ref(
        d_nv_.signal, d_nv_.reference,
        d_n0_.signal, d_n0_.reference
    )
    return d_nv_.sweep_tus * 1000, contrast_c   # (ns, %)

# Uncomment to start:
# qd.live_plot(get_corrected_rabi)

---
## Section 2: Vector Magnetometry Theory

### NV Crystallographic Axes

Four NV orientations along ⟨111⟩ directions:

$$
\hat{n}_1 = \frac{1}{\sqrt{3}}[1,1,1], \quad
\hat{n}_2 = \frac{1}{\sqrt{3}}[1,-1,-1], \quad
\hat{n}_3 = \frac{1}{\sqrt{3}}[-1,1,-1], \quad
\hat{n}_4 = \frac{1}{\sqrt{3}}[-1,-1,1]
$$

Each axis gives two ESR transitions → **8 ODMR peaks** total.

### Field Reconstruction

Frequency shifts $\boldsymbol{\delta\nu} = \gamma_e A\,\delta\mathbf{B}$.  
Inverse (pseudo-inverse, Eq. 8):

$$\delta\mathbf{B} = \frac{1}{\gamma_e} T\,\boldsymbol{\delta\nu}, \qquad
T = \frac{\sqrt{3}}{4} \begin{pmatrix} 1 & 1 & -1 & -1 \\ 1 & -1 & 1 & -1 \\ 1 & -1 & -1 & 1 \end{pmatrix}$$

**With noise cancellation**, the peak frequency extraction is performed on the noise-corrected ODMR spectrum, reducing the peak position uncertainty $\sigma_\nu$ and therefore the field uncertainty $\sigma_B \propto \sigma_\nu$.

In [ ]:
# NV axes and transformation matrices
nv_axes = np.array([
    [ 1,  1,  1],
    [ 1, -1, -1],
    [-1,  1, -1],
    [-1, -1,  1]
]) / np.sqrt(3)

GAMMA_E = 28.0    # GHz/T
D_ZFS   = 2.87    # GHz

A_forward = GAMMA_E * nv_axes   # (4×3)

s = np.sqrt(3) / 4 / GAMMA_E
T_inverse = s * np.array([   # (3×4)
    [ 1,  1, -1, -1],
    [ 1, -1,  1, -1],
    [ 1, -1, -1,  1]
])

print("Verification T @ A (should be identity):")
print(np.round(T_inverse @ A_forward, 4))

---
## Section 3: Noise-Corrected Wide ODMR — All 8 NV Peaks

Run LockinODMR on both channels over the full bias-field range.  
The noise-corrected spectrum gives lower noise floors, improving detection of weak or overlapping peaks.

In [ ]:
# ── Wide ODMR configuration ────────────────────────────────────────────────
def make_wide_odmr_config(adc_ch, start_MHz=2600, stop_MHz=3200, delta_MHz=1, reps=20):
    cfg = make_config(adc_ch)
    cfg.readout_integration_tus = qd.max_int_time_tus
    cfg.mw_gain                 = 10000
    cfg.pre_init                = True
    cfg.reps                    = reps
    cfg.relax_delay_treg        = 500
    cfg.add_linear_sweep('mw', 'fMHz', start=start_MHz, stop=stop_MHz, delta=delta_MHz)
    return cfg

# ===== EDIT: adjust to match your bias field strength =====
WIDE_START_MHZ = 2600
WIDE_STOP_MHZ  = 3200
WIDE_REPS      = 20

prog_wide_nv = qd.LockinODMR(make_wide_odmr_config(NV_CHANNEL,    WIDE_START_MHZ, WIDE_STOP_MHZ, reps=WIDE_REPS))
prog_wide_n0 = qd.LockinODMR(make_wide_odmr_config(NOISE_CHANNEL, WIDE_START_MHZ, WIDE_STOP_MHZ, reps=WIDE_REPS))

print(f"Wide ODMR: {WIDE_START_MHZ} → {WIDE_STOP_MHZ} MHz, {WIDE_REPS} reps")
print(f"  Estimated time (NV ch): {prog_wide_nv.total_time():.1f} s")
print()
print("[1/2] Acquiring wide ODMR on NV channel...")
d_wide_nv = prog_wide_nv.acquire(progress=True)
print("[2/2] Acquiring wide ODMR on noise channel...")
d_wide_n0 = prog_wide_n0.acquire(progress=True)
print("Done.")

In [ ]:
# ── Apply noise correction ─────────────────────────────────────────────────
_, _, wide_contrast_c = correct_signal_ref(
    d_wide_nv.signal, d_wide_nv.reference,
    d_wide_n0.signal, d_wide_n0.reference
)

wide_freqs = d_wide_nv.frequencies

print(f"Wide ODMR noise floor — raw:       {np.std(d_wide_nv.contrast_percent):.4f} %")
print(f"Wide ODMR noise floor — corrected: {np.std(wide_contrast_c):.4f} %")

In [ ]:
# ── Peak detection on noise-corrected spectrum ─────────────────────────────
def find_nv_peaks(frequencies, contrast_percent, n_peaks=8, prominence_factor=1.5):
    """Find ODMR dip positions. Returns (peak_indices, peak_frequencies) sorted by frequency."""
    inverted  = -contrast_percent
    threshold = np.std(inverted) * prominence_factor
    min_dist  = max(1, len(frequencies) // (n_peaks * 3))
    idx, _    = scipy_find_peaks(inverted, height=threshold, distance=min_dist)
    order     = np.argsort(frequencies[idx])
    idx       = idx[order]
    return idx, frequencies[idx]

peak_idx, peak_freqs = find_nv_peaks(wide_freqs, wide_contrast_c, n_peaks=8)

fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

ax = axes[0]
ax.plot(wide_freqs, d_wide_nv.contrast_percent, color='steelblue', linewidth=1, alpha=0.6, label='Raw (Ch1)')
ax.plot(wide_freqs, wide_contrast_c,            color='green',     linewidth=1.5,           label='Noise-corrected')
ax.plot(wide_freqs[peak_idx], wide_contrast_c[peak_idx], 'rv', markersize=10, label=f'{len(peak_freqs)} peaks')
for i, (idx, freq) in enumerate(zip(peak_idx, peak_freqs)):
    ax.annotate(f'{i+1}\n{freq:.1f}', xy=(freq, wide_contrast_c[idx]),
                xytext=(0, -22), textcoords='offset points', ha='center', fontsize=8, color='darkred')
ax.axvline(D_ZFS * 1000, color='gray', linestyle='--', alpha=0.5, label='D = 2870 MHz')
ax.axhline(0, color='k', linewidth=0.5, linestyle='--')
ax.set_ylabel('Contrast (%)')
ax.set_title(f'Wide ODMR — noise-corrected — {len(peak_freqs)} peaks detected (8 expected)')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

ax = axes[1]
ax.plot(wide_freqs, d_wide_n0.contrast_percent, color='darkorange', linewidth=1.2,
        label=f'Noise ref (Ch{NOISE_CHANNEL}) — ideally flat')
ax.axhline(0, color='k', linewidth=0.5, linestyle='--')
ax.set_ylabel('Contrast (%)')
ax.set_xlabel('Frequency (MHz)')
ax.set_title('Noise Reference Channel — Wide ODMR')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Detected peaks (noise-corrected):")
for i, f in enumerate(peak_freqs):
    print(f"  Peak {i+1}: {f:.2f} MHz")

if len(peak_freqs) < 8:
    print(f"\nWARNING: Only {len(peak_freqs)} peaks found. Try increasing reps or mw_gain.")

---
## Section 4: Peak Assignment to NV Crystallographic Axes

Pair the 8 noise-corrected peaks into 4 NV axis pairs by matching outer-to-inner.  
Pairs bracket D = 2870 MHz symmetrically; largest splitting = axis most aligned with bias field.

In [ ]:
def assign_peaks_to_axes(peak_frequencies):
    """Pair 8 ODMR peaks into 4 NV axis pairs (outer-to-inner).
    Returns (pairs, splittings, centers) sorted by splitting.
    """
    freqs   = np.sort(peak_frequencies)
    n_pairs = len(freqs) // 2
    pairs, splittings, centers = [], [], []
    for i in range(n_pairs):
        f_lo = freqs[i]; f_hi = freqs[len(freqs) - 1 - i]
        pairs.append((f_lo, f_hi))
        splittings.append(f_hi - f_lo)
        centers.append((f_lo + f_hi) / 2)
    order      = np.argsort(splittings)
    pairs      = [pairs[i] for i in order]
    splittings = np.array([splittings[i] for i in order])
    centers    = np.array([centers[i]    for i in order])
    return pairs, splittings, centers


if len(peak_freqs) == 8:
    pairs, splittings, centers = assign_peaks_to_axes(peak_freqs)
    print(f"{'Axis':>5} | {'f_low (MHz)':>12} | {'f_high (MHz)':>12} | {'Split (MHz)':>12} | {'B_proj (mT)':>11}")
    print("-" * 65)
    for i, ((f_lo, f_hi), split, ctr) in enumerate(zip(pairs, splittings, centers)):
        B_proj = split / (2 * GAMMA_E * 1e3) * 1e3
        print(f"  {i+1:>3} | {f_lo:>12.2f} | {f_hi:>12.2f} | {split:>12.2f} | {B_proj:>11.3f}")
    print(f"\nBias field ≈ {splittings[-1]/(2*GAMMA_E*1e3)*1e3:.2f} mT (from largest splitting)")
else:
    print(f"Need 8 peaks; found {len(peak_freqs)}. Re-run wide ODMR.")

In [ ]:
# Visualise peak pairs on noise-corrected spectrum
if len(peak_freqs) == 8:
    colors = ['steelblue', 'darkorange', 'green', 'red']
    plt.figure(figsize=(14, 4))
    plt.plot(wide_freqs, wide_contrast_c, 'k-', linewidth=1, alpha=0.5)
    for i, (f_lo, f_hi) in enumerate(pairs):
        plt.axvspan(f_lo, f_hi, alpha=0.15, color=colors[i],
                    label=f'Axis {i+1}  Δ={splittings[i]:.1f} MHz')
        for f in [f_lo, f_hi]:
            idx_c = np.argmin(np.abs(wide_freqs - f))
            plt.plot(f, wide_contrast_c[idx_c], 'v', color=colors[i], markersize=10)
    plt.axvline(D_ZFS * 1000, color='gray', linestyle='--', alpha=0.5, label='D = 2870 MHz')
    plt.axhline(0, color='k', linewidth=0.5, linestyle='--')
    plt.xlabel('Frequency (MHz)')
    plt.ylabel('Contrast (%)')
    plt.title('Noise-Corrected ODMR — Peaks Grouped by NV Axis')
    plt.legend(loc='upper right', fontsize=8)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

---
## Section 5: Frequency Shift Tracking & Field Reconstruction

**Reference** → noise-corrected ODMR with stable bias field.  
**Measurement** → noise-corrected ODMR with unknown field.  
**Field vector** → $\delta\mathbf{B} = T\,\boldsymbol{\delta\nu} / \gamma_e$.

Noise cancellation is applied inside every call to `acquire_corrected_odmr()`,
so peak extraction always uses the noise-suppressed spectrum.

In [ ]:
# ── Build reference ODMR programs (reused for every measurement) ───────────
def make_ref_odmr_config(adc_ch, reps=50):
    cfg = make_config(adc_ch)
    cfg.readout_integration_tus = qd.max_int_time_tus
    cfg.mw_gain                 = 10000
    cfg.pre_init                = True
    cfg.reps                    = reps
    cfg.relax_delay_treg        = 500
    cfg.add_linear_sweep('mw', 'fMHz', start=WIDE_START_MHZ, stop=WIDE_STOP_MHZ, delta=1)
    return cfg

prog_ref_nv = qd.LockinODMR(make_ref_odmr_config(NV_CHANNEL,    reps=50))
prog_ref_n0 = qd.LockinODMR(make_ref_odmr_config(NOISE_CHANNEL, reps=50))

print("Reference ODMR programs built.")
print(f"  Estimated time (NV ch): {prog_ref_nv.total_time():.1f} s")

In [ ]:
# Step 1: Reference measurement
print("=== Reference Measurement (noise-corrected) ===")
print("Ensure the bias field is stable and laser is ON.")
print()

print("[1/2] Acquiring reference on NV channel...")
d_ref_nv = prog_ref_nv.acquire(progress=True)
print("[2/2] Acquiring reference on noise channel...")
d_ref_n0 = prog_ref_n0.acquire(progress=True)

_, _, ref_contrast_c = correct_signal_ref(
    d_ref_nv.signal, d_ref_nv.reference,
    d_ref_n0.signal, d_ref_n0.reference
)

ref_idx, ref_peak_freqs = find_nv_peaks(d_ref_nv.frequencies, ref_contrast_c, n_peaks=8)
ref_pairs, ref_splittings, ref_centers = assign_peaks_to_axes(ref_peak_freqs)

print(f"\nReference peaks ({len(ref_peak_freqs)} found, noise-corrected):")
for i, f in enumerate(ref_peak_freqs):
    print(f"  Peak {i+1}: {f:.2f} MHz")

In [ ]:
# ── Field measurement function (noise-corrected) ───────────────────────────
def measure_field_change(prog_nv, prog_n0, ref_pairs, ref_centers):
    """Acquire one noise-corrected ODMR and compute 3D field change.

    Returns dict with field_vector_uT, freq_shifts_MHz, and raw data.
    Returns None if peak detection fails.
    """
    d_nv_ = prog_nv.acquire(progress=False)
    d_n0_ = prog_n0.acquire(progress=False)

    _, _, contrast_c = correct_signal_ref(
        d_nv_.signal, d_nv_.reference,
        d_n0_.signal, d_n0_.reference
    )

    cur_idx, cur_peak_freqs = find_nv_peaks(d_nv_.frequencies, contrast_c, n_peaks=8)
    if len(cur_peak_freqs) != 8:
        print(f"Warning: found {len(cur_peak_freqs)} peaks (expected 8). Returning None.")
        return None

    cur_pairs, cur_splittings, cur_centers = assign_peaks_to_axes(cur_peak_freqs)
    freq_shifts_MHz = cur_centers - ref_centers
    freq_shifts_GHz = freq_shifts_MHz / 1000
    field_vector_T  = T_inverse @ freq_shifts_GHz

    return {
        'field_vector_T':   field_vector_T,
        'field_vector_uT':  field_vector_T * 1e6,
        'freq_shifts_MHz':  freq_shifts_MHz,
        'current_peaks':    cur_peak_freqs,
        'frequencies':      d_nv_.frequencies,
        'contrast_percent': contrast_c,
        'contrast_raw':     d_nv_.contrast_percent,
    }

In [ ]:
# Step 2: Single field measurement
print("=== Field Measurement (noise-corrected) ===")

result = measure_field_change(prog_ref_nv, prog_ref_n0, ref_pairs, ref_centers)

if result is not None:
    print(f"\nFrequency shifts per axis (MHz):")
    for i, shift in enumerate(result['freq_shifts_MHz']):
        print(f"  Axis {i+1}: {shift:+.3f} MHz")

    dB = result['field_vector_uT']
    print(f"\n3D Field change (noise-corrected, relative to reference):")
    print(f"  ΔBx = {dB[0]:+.3f} μT")
    print(f"  ΔBy = {dB[1]:+.3f} μT")
    print(f"  ΔBz = {dB[2]:+.3f} μT")
    print(f"  |ΔB| = {np.linalg.norm(dB):.3f} μT")

In [ ]:
# Compare reference vs current noise-corrected ODMR
if result is not None:
    fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

    ax = axes[0]
    ax.plot(d_ref_nv.frequencies, ref_contrast_c,
            'b-', alpha=0.8, linewidth=1.2, label='Reference (corrected)')
    ax.plot(result['frequencies'], result['contrast_percent'],
            'r-', alpha=0.8, linewidth=1.2, label='Current (corrected)')
    ax.axhline(0, color='k', linewidth=0.5, linestyle='--')
    ax.set_ylabel('Contrast (%)')
    ax.set_title('Noise-Corrected ODMR: Reference vs Current')
    ax.legend()
    ax.grid(True, alpha=0.3)

    ax = axes[1]
    ax.plot(result['frequencies'], result['contrast_raw'],
            color='steelblue', linewidth=1, alpha=0.6, label='Raw (Ch1 only)')
    ax.plot(result['frequencies'], result['contrast_percent'],
            color='green', linewidth=1.4, label='Noise-corrected')
    ax.axhline(0, color='k', linewidth=0.5, linestyle='--')
    ax.set_ylabel('Contrast (%)')
    ax.set_title('Current Measurement: Raw vs Noise-Corrected')
    ax.legend()
    ax.grid(True, alpha=0.3)

    ref_interp = np.interp(result['frequencies'], d_ref_nv.frequencies, ref_contrast_c)
    ax = axes[2]
    ax.plot(result['frequencies'], result['contrast_percent'] - ref_interp,
            color='purple', linewidth=1.2)
    ax.axhline(0, color='k', linewidth=0.5, linestyle='--')
    ax.set_ylabel('Δ Contrast (%)')
    ax.set_xlabel('Frequency (MHz)')
    ax.set_title('ODMR Difference (Current − Reference) — sensitive to field change')
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

---
## Section 6: Drift Rejection via Field Flipping (Optional)

Cancels slow bias-field drift by flipping polarity:
$$\boldsymbol{\delta\nu}_{\text{local}} = \frac{\boldsymbol{\delta\nu}^+ - \boldsymbol{\delta\nu}^-}{2}$$

Both $\boldsymbol{\delta\nu}^+$ and $\boldsymbol{\delta\nu}^-$ are extracted from noise-corrected spectra.

In [ ]:
def measure_with_drift_rejection(prog_nv, prog_n0, ref_pairs, ref_centers):
    """Noise-corrected field measurement with bias-field drift rejection."""
    print("Step 1: Bias field at +B₀")
    input("Press Enter when ready...")
    result_pos = measure_field_change(prog_nv, prog_n0, ref_pairs, ref_centers)
    if result_pos is None:
        return None
    print(f"  +B₀ shifts: {result_pos['freq_shifts_MHz']}")

    print("\nStep 2: Flip bias field to −B₀")
    input("Press Enter when ready...")
    result_neg = measure_field_change(prog_nv, prog_n0, ref_pairs, ref_centers)
    if result_neg is None:
        return None
    print(f"  −B₀ shifts: {result_neg['freq_shifts_MHz']}")

    local_shifts_MHz = (result_pos['freq_shifts_MHz'] - result_neg['freq_shifts_MHz']) / 2
    local_field_T    = T_inverse @ (local_shifts_MHz / 1000)
    local_field_uT   = local_field_T * 1e6

    print(f"\nDrift-rejected + noise-corrected local field:")
    print(f"  ΔBx = {local_field_uT[0]:+.3f} μT")
    print(f"  ΔBy = {local_field_uT[1]:+.3f} μT")
    print(f"  ΔBz = {local_field_uT[2]:+.3f} μT")
    print(f"  |ΔB| = {np.linalg.norm(local_field_uT):.3f} μT")
    return local_field_uT


# Uncomment to run (requires interactive or automated bias field control):
# local_field = measure_with_drift_rejection(prog_ref_nv, prog_ref_n0, ref_pairs, ref_centers)

---
## Section 7: Error Estimation

Field uncertainty from per-axis frequency uncertainty $\sigma_\nu$:
$$\sigma_B = \frac{3}{16 \gamma_e^2} \sum_{i=1}^{4} \sigma_{\nu_i}^2$$

Noise cancellation reduces the effective $\sigma_\nu$ (lower noise floor → sharper peak centroid).  
The improvement factor can be estimated from the noise reduction measured in Notebook 1b, Section 3.

In [ ]:
def estimate_field_uncertainty(freq_uncertainties_MHz):
    """Field uncertainty from per-axis frequency uncertainties (Eq. 11)."""
    sigma_nu_GHz = np.asarray(freq_uncertainties_MHz) / 1000
    return (3 / (16 * GAMMA_E**2)) * np.sum(sigma_nu_GHz**2) * 1e6   # μT


# ===== EDIT: set your estimated ODMR linewidth and SNR =====
odmr_linewidth_MHz    = 5.0    # FWHM in MHz
snr_raw               = 10.0   # SNR without noise cancellation
# Noise reduction from Notebook 1b calibration (e.g. 40 % reduction → factor 0.6)
noise_reduction_factor = 0.6   # <-- update from 01b Section 3 output

freq_unc_raw  = odmr_linewidth_MHz / snr_raw
freq_unc_corr = freq_unc_raw * noise_reduction_factor   # improved by noise cancellation

sigma_B_raw  = estimate_field_uncertainty(np.full(4, freq_unc_raw))
sigma_B_corr = estimate_field_uncertainty(np.full(4, freq_unc_corr))

print(f"ODMR linewidth (FWHM):       {odmr_linewidth_MHz:.1f} MHz")
print(f"SNR estimate (raw):          {snr_raw:.1f}")
print(f"Noise reduction factor:      {noise_reduction_factor:.2f}")
print()
print(f"Frequency uncertainty — raw:       {freq_unc_raw:.3f} MHz / axis")
print(f"Frequency uncertainty — corrected: {freq_unc_corr:.3f} MHz / axis")
print()
print(f"Field uncertainty — raw:           {sigma_B_raw:.4f} μT")
print(f"Field uncertainty — corrected:     {sigma_B_corr:.4f} μT")
print(f"Improvement factor:                {sigma_B_raw / sigma_B_corr:.2f}×")

---
## Section 8: Live Noise-Corrected Field Monitoring

Continuously acquires noise-corrected ODMR and tracks the 3D field vector.  
Each loop iteration acquires both channels, applies correction, fits peaks, and returns |ΔB|.

In [ ]:
field_history = []

def get_corrected_field_magnitude():
    result = measure_field_change(prog_ref_nv, prog_ref_n0, ref_pairs, ref_centers)
    if result is not None:
        field_history.append(result['field_vector_uT'])
        return np.linalg.norm(result['field_vector_uT'])
    return 0.0

# Uncomment to start:
# print("Live |ΔB| monitoring (noise-corrected, μT). Press Stop to exit.")
# qd.live_plot(get_corrected_field_magnitude)

In [ ]:
# Plot field vector history after live monitoring
if len(field_history) > 0:
    history = np.array(field_history)
    t_axis  = np.arange(len(history))

    fig, axes = plt.subplots(4, 1, figsize=(12, 10), sharex=True)
    labels = ['ΔBx', 'ΔBy', 'ΔBz', '|ΔB|']
    colors = ['steelblue', 'darkorange', 'green', 'red']

    for i in range(3):
        axes[i].plot(t_axis, history[:, i], color=colors[i], linewidth=1.2)
        axes[i].set_ylabel(f'{labels[i]} (μT)')
        axes[i].axhline(0, color='k', linewidth=0.5, linestyle='--')
        axes[i].grid(True, alpha=0.3)

    mag = np.linalg.norm(history, axis=1)
    axes[3].plot(t_axis, mag, color=colors[3], linewidth=1.2)
    axes[3].set_ylabel('|ΔB| (μT)')
    axes[3].set_xlabel('Measurement index')
    axes[3].grid(True, alpha=0.3)

    axes[0].set_title('Noise-Corrected Field Vector History')
    plt.tight_layout()
    plt.show()

    print(f"Field statistics over {len(history)} measurements:")
    for i, label in enumerate(['Bx', 'By', 'Bz']):
        print(f"  Δ{label}: mean = {np.mean(history[:,i]):+.3f} μT,  std = {np.std(history[:,i]):.3f} μT")
    print(f"  |ΔB|:  mean = {np.mean(mag):.3f} μT,  std = {np.std(mag):.3f} μT")
else:
    print("No field history to plot. Run the live monitoring cell first.")

---
## Summary: Complete Noise-Corrected Measurement Workflow

| Step | Notebook | What you get |
|------|----------|--------------|
| 1. Connect & verify | 01b Sec 0 | RFSoC connection |
| 2. Dual-channel PL | 01b Sec 2 | Both channels live |
| 3. Noise calibration | 01b Sec 3 | **α** — laser noise gain |
| 4. Corrected ODMR | 01b Sec 5 | Resonance freq, improved SNR |
| 5. Corrected readout window | 01b Sec 6 | τ, integration time |
| 6. Corrected Rabi | 02b Sec 1 | π/2 pulse length |
| 7. Corrected wide ODMR | 02b Sec 3 | All 8 peaks, lower noise floor |
| 8. Reference measurement | 02b Sec 5 | Calibration state |
| 9. Field measurement | 02b Sec 5 | Noise-corrected 3D field |
| 10. Drift rejection | 02b Sec 6 | Bias-drift + laser-noise rejected |
| 11. Error estimation | 02b Sec 7 | σ_B improvement quantified |

### Key variables carried throughout
```python
ALPHA                    # noise gain: ch1_corrected = ch1 - ALPHA * ch0
correct_signal_ref()     # applies correction to any (signal, reference) pair
acquire_corrected_odmr() # convenience wrapper for dual-channel LockinODMR
measure_field_change()   # full noise-corrected field vector acquisition
```

### Further sensitivity improvements
- Use **Ramsey spectroscopy** (`qd.Ramsey`) instead of ODMR for narrower linewidth (limited by T₂* rather than MW power)
- Increase `reps` for more averaging
- Re-calibrate **α** if optics are realigned